In [203]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [204]:
df=pd.read_csv("Titanic-Dataset.csv")
y_train=df["Survived"]
X_train=df.drop(columns=["Survived"])

In [205]:
def feature_engineering(new_df):
    df=new_df.copy()
    df["individual_fare"]=df["Fare"]/(df["SibSp"]+df["Parch"]+1)
    df["Cabin"]=df["Cabin"].fillna("M")
    df["Deck"]=df["Cabin"].str.get(0)
    df["FamilySize"]=df["Parch"]+df["SibSp"]+1
    
    bin_size=[-1,1,4,6,15]
    size_labels = ['single', 'small', 'medium', 'large']
    df['Group_Size'] = pd.cut(df['FamilySize'], bins=bin_size, labels=size_labels)
    
    df.drop(columns=["Name","Ticket","Fare","Cabin","SibSp","Parch","FamilySize"],inplace=True)
    
    
    return df
feature_transformer1 = FunctionTransformer(
    feature_engineering,
    validate=False
)

In [206]:

embarked_encoder=OrdinalEncoder(
    categories=[["M", "Q", "S", "C"]]
)

from sklearn.pipeline import Pipeline

embarked_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="M")),
    ("encoder", OrdinalEncoder(categories=[["M", "Q", "S", "C"]]))
])

age_pipeline = Pipeline([
    ("imputer", KNNImputer(weights="distance", n_neighbors=2))
])



trf = ColumnTransformer([
    ("Age", age_pipeline, ["Age"]),
    ("Embarked", embarked_pipeline, ["Embarked"]),
    ("Sex", OrdinalEncoder(), ["Sex"]),
    ("Deck", OneHotEncoder(sparse_output=False,handle_unknown="ignore"), ["Deck"]),
    ("Group_Size", OrdinalEncoder(), ["Group_Size"])
], remainder="passthrough")


pipeline=Pipeline([
    ("feature_engineering",feature_transformer1),
    ("trf",trf),
    ("model",RandomForestClassifier(n_estimators=200,criterion="entropy",max_depth=25))
])

In [207]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [10, 20, 30, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
    "model__criterion": ["gini", "entropy"]
}

In [208]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    error_score="raise"
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


Python(62028) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62029) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62030) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62031) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62032) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62033) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62034) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62035) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.1s
[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.2s
[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.1s
[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.2s
[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.2s
[CV] END model__criterion=gini, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=1, mode

In [209]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 10, 'model__n_estimators': 300}
0.8372795179210344


In [210]:
best_model = grid_search.best_estimator_



In [211]:
test_df=pd.read_csv("tested.csv")
y_test=test_df["Survived"]
X_test=test_df.drop(columns=["Survived"])
y_pred=best_model.predict(X_test)
accuracy_=accuracy_score(y_test,y_pred)
print(f"test accuracy: {accuracy_}")

test accuracy: 0.8373205741626795
